# 20. Data Cleaning, Filtering, and Deduplication

## Problem

为什么原始抓取数据不能直接拿去训练？数据清洗、质量过滤、完全去重、近重复检测和质量分桶分别在解决什么问题？

## Dependencies

建议先读完数据采集那一节，因为这一节处理的对象，就是上一阶段产生的原始记录。


## Why raw data is dangerous

原始抓取数据常常混着大量噪声：

- HTML 标签和脚本碎片
- 导航栏、广告、法律页脚
- 极短文本和无意义模板文本
- 重复页面、镜像页面、转载页面
- 乱码、多语言错分、抽取失败样本

如果这些内容不经过处理就直接进训练，模型就会把很多无效模式也当成语言规律去学习。结果通常有两类：

- 训练效率下降，因为大量 token 根本没有价值
- 输出质量下降，因为模型学到了模板噪声、广告语、碎片结构

所以清洗不是一个“后处理小步骤”，而是决定数据底座质量的核心环节。


## A practical cleaning pipeline

把这一阶段拆开看，通常至少会包含下面几层：

1. **Normalization**：统一文本格式，清掉明显标签残留和异常空白。
2. **Basic filtering**：先排掉极短文本、广告文本、乱码文本、明显错误文本。
3. **Exact deduplication**：完全一样的内容只保留一份。
4. **Near-duplicate detection**：模板页、镜像页、转载页做相似度级别的去重。
5. **Quality bucketing**：按质量高低、来源、长度、风格做分桶。

真正重要的是理解：这些步骤不是同义词，它们在解决不同层次的问题。


In [ ]:
# 先构造一批 toy 原始记录。
# 这些样本故意包含 HTML、广告语、重复文本和近重复文本。

raw_texts = [
    '<html><body>Buy now!<div>Deep learning tutorial</div></body></html>',
    'Deep learning tutorial',
    'Deep learning tutorial!!!',
    'Deep learning tutorial for beginners',
    'Cookie settings Privacy Terms',
    'ok',
]


def normalize_text(text):
    # 这里只做极简示意。真实系统会用更稳的 HTML 清理和 Unicode 规范化。
    text = text.replace('<html>', '').replace('</html>', '')
    text = text.replace('<body>', '').replace('</body>', '')
    text = text.replace('<div>', ' ').replace('</div>', ' ')
    text = text.replace('!!!', '!')
    return ' '.join(text.split()).strip()


normalized = [normalize_text(text) for text in raw_texts]
print('normalized =')
for item in normalized:
    print(' -', item)


## Basic filtering is not the same as deduplication

清洗第一层经常是过滤，而不是去重。因为有些样本的问题不是“它和别人重复”，而是“它自己就不该存在”。

例如：

- 太短
- 广告味太重
- 全是模板词
- 抽取失败后只剩导航或页脚

这类样本即使只出现一次，也应该先被排掉。


In [ ]:
# 这里写一个最小质量过滤器。
# 规则非常粗糙，但足以说明过滤和去重是两个不同步骤。

def simple_quality_filter(text):
    lowered = text.lower()
    if len(text) < 10:
        return False
    if 'buy now' in lowered:
        return False
    if 'privacy terms' in lowered:
        return False
    return True

filtered = [text for text in normalized if simple_quality_filter(text)]
print('after filtering =')
for item in filtered:
    print(' -', item)


## Exact duplicates vs near duplicates

很多人会把“去重”只理解成删掉完全一样的字符串。这只解决了一半问题。

### Exact duplicates

如果两条文本逐字符完全一样，那很容易处理：保留一份即可。

### Near duplicates

真正更麻烦的是：

- 同一篇文章改了标题
- 教程只多加一句话
- 模板页只改了几个字段
- 镜像站转载后只换了页脚

这些文本并不完全相同，但它们的训练价值高度重叠。如果大量保留，会让模型反复见同一个模式，浪费训练预算，还会放大某些模板分布。


In [ ]:
# 先做完全去重。
# dict.fromkeys 可以保留首次出现顺序，同时去掉完全重复样本。

exact_deduped = list(dict.fromkeys(filtered))
print('after exact dedup =')
for item in exact_deduped:
    print(' -', item)


def jaccard_similarity(a, b):
    # 这里用 token 集合重叠度做一个最简单的近重复示意。
    # 真实大规模系统通常会进一步上 shingling、MinHash、LSH。
    a_tokens = set(a.lower().split())
    b_tokens = set(b.lower().split())
    return len(a_tokens & b_tokens) / max(1, len(a_tokens | b_tokens))


near_dup_threshold = 0.75
print('
near-duplicate checks =')
for i in range(len(exact_deduped)):
    for j in range(i + 1, len(exact_deduped)):
        left = exact_deduped[i]
        right = exact_deduped[j]
        sim = jaccard_similarity(left, right)
        print(f'similarity={sim:.4f} | near_dup={sim >= near_dup_threshold} | {left!r} <-> {right!r}')


## Why threshold selection is tricky

近重复阈值不是一个随便拍脑袋的数字，它直接影响你留下什么、删掉什么。

- 阈值太低：很多本来只是主题相近、但内容仍有增量的文本会被误删。
- 阈值太高：大量模板页、转载页、镜像页仍然会漏过去。

所以真实工程里，阈值通常会和来源类型一起调：

- 文档站可能更适合更严格的近重复策略
- 论坛讨论可能要更宽松，因为相似问题下的答案仍然可能有增量

这也是为什么“去重策略”不能脱离 source type 单独谈。


In [ ]:
# 再做一个最小质量分桶示意。
# 重点不是这个函数本身，而是理解：清洗后通常不会所有文本直接一锅进训练。

def quality_bucket(text):
    lowered = text.lower()
    if len(text) > 30 and 'tutorial' in lowered:
        return 'high_quality'
    if len(text) > 12:
        return 'medium_quality'
    return 'review_or_drop'

bucketed = [(text, quality_bucket(text)) for text in exact_deduped]
print('quality buckets =')
for item in bucketed:
    print(item)


## Why quality buckets matter for training

清洗后的文本通常不会立刻全部平权混在一起。常见做法是先分桶，例如：

- 高质量技术文档
- 中等质量网页正文
- 讨论类文本
- 待复查文本

这样做的价值在于：

- 不同桶可以有不同采样权重
- 某些阶段可以更偏向高质量桶
- 后面如果发现某个桶问题大，也更容易整体回滚或重清洗

所以 quality bucket 不只是一个标签，它会直接影响训练分布。


In [ ]:
# 最后把整个 toy pipeline 串起来，看“原始记录 -> 可训练文本”的流转。

pipeline_summary = {
    'raw_count': len(raw_texts),
    'normalized_count': len(normalized),
    'filtered_count': len(filtered),
    'exact_deduped_count': len(exact_deduped),
    'bucketed_count': len(bucketed),
}

print('pipeline summary =')
for key, value in pipeline_summary.items():
    print(f'{key}: {value}')


## Common mistakes

- 以为做一次正则替换就叫数据清洗。
- 只做完全去重，不做近重复检测。
- 过滤太松，让模板噪声和广告文本大量漏进训练集。
- 过滤太严，把稀缺但有价值的长尾数据也删掉了。
- 不分桶，导致高质量和低质量语料在训练时被同样对待。

## Summary

这一节最重要的是看清楚：数据清洗、过滤和去重不是一个动作，而是一条分层处理链。它的目标不是把文本“洗漂亮”，而是尽可能把训练预算留给真正有价值的模式，把噪声、重复和模板污染挡在训练入口之外。
